In [111]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import joblib, os

In [112]:
df=pd.read_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\final_processed_data.csv')
df1=pd.read_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\batsman_profiles.csv')

In [113]:
df['Batsman_Name_clean'] = df['Full Name'].astype(str).str.lower().str.strip()
df1['Full_Name_clean'] = df1['Full Name'].astype(str).str.lower().str.strip()

In [114]:
for col in ['isFour', 'isSix', 'isWicket']:
    df[col] = df[col].astype(int)


In [115]:
df['run'] = pd.to_numeric(df['run'], errors='coerce').fillna(0)

In [116]:
df['pitchLine_code'] = df['pitchLine'].astype('category').cat.codes
df['pitchLength_code'] = df['pitchLength'].astype('category').cat.codes

In [117]:
pitchLine_mapping = dict(enumerate(df['pitchLine'].astype('category').cat.categories))
pitchLength_mapping = dict(enumerate(df['pitchLength'].astype('category').cat.categories))


In [118]:
df['isBoundary'] = df['isFour'] | df['isSix'] 

In [119]:
df.columns

Index(['Unnamed: 0', 'match_obj_id', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Full Name', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Full Name_bowler', 'Batting Style (s)_bowler', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role', 'Batsman_Name_clean', 'pitchLine_code',
       'pitchLength_code', 'isBoundary'],
      dtype='object')

## Filtering Data for Batsmen

This cell filters the dataset to include only batsman-related records.  
It removes unnecessary or incomplete data to make the model more accurate and efficient.


In [120]:
agg_cols = ['run', 'isFour', 'isSix', 'isWicket', 'isBoundary', 'pitchLine_code', 'pitchLength_code']
batsman_stats = df.groupby('Batsman_Name_clean')[agg_cols].agg({
    'run': ['sum', 'mean'],
    'isFour': 'sum',
    'isSix': 'sum',
    'isWicket': 'sum',
    'isBoundary': 'sum',
    'pitchLine_code': lambda x: x.mode()[0] if not x.mode().empty else np.nan,
    'pitchLength_code': lambda x: x.mode()[0] if not x.mode().empty else np.nan
}).reset_index()

In [121]:
batsman_stats.columns = [
    'Batsman_Name_clean',
    'total_runs', 'avg_runs', 'total_4s', 'total_6s',
    'total_wickets', 'total_boundaries', 'mode_pitchLine', 'mode_pitchLength'
]

In [122]:
balls_faced = df.groupby('Batsman_Name_clean').size().rename('balls_faced')
batsman_stats = batsman_stats.merge(balls_faced, on='Batsman_Name_clean', how='left')

In [123]:
batsman_stats['strike_rate'] = batsman_stats['total_runs'] / batsman_stats['balls_faced'] * 100
batsman_stats['boundary_rate'] = batsman_stats['total_boundaries'] / batsman_stats['balls_faced']


In [124]:
batsman_stats['Batsman_Name_clean']

0         aamir kaleem
1          aaron finch
2        aaron johnson
3          aaron jones
4       aaron phangiso
            ...       
457       yuvraj singh
458        zahoor khan
459         zane green
460        zawar farid
461    zeeshan maqsood
Name: Batsman_Name_clean, Length: 462, dtype: object

In [125]:
batsman_stats['dot_ball_pct'] = df.groupby('Batsman_Name_clean')['run'].apply(lambda x: (x==0).sum() / x.count() * 100)


In [126]:
batsman_stats['dismissal_ratio'] = batsman_stats['total_wickets'] / batsman_stats['balls_faced']

In [127]:
df_features = pd.merge(
    df1,
    batsman_stats,
    left_on='Full_Name_clean',
    right_on='Batsman_Name_clean',
    how='left'
)

In [128]:
def get_phase(over):
    if over <= 6:
        return 'Powerplay'
    elif over <= 15:
        return 'Middle'
    else:
        return 'Death'

df['phase'] = df['oversActual'].apply(get_phase)

phase_perf = df.groupby(['Batsman_Name_clean', 'phase'])['run'].mean().unstack().fillna(0)
phase_perf.columns = [f'avg_runs_{c.lower()}' for c in phase_perf.columns]
batsman_stats = batsman_stats.merge(phase_perf, left_index=True, right_index=True, how='left')


In [129]:
batsman_stats['dismissal_ratio'] = batsman_stats['total_wickets'] / batsman_stats['balls_faced']


In [130]:
consistency = df.groupby('Batsman_Name_clean')['run'].std().rename('run_std')
batsman_stats = batsman_stats.merge(consistency, left_index=True, right_index=True, how='left')
batsman_stats['consistency_index'] = batsman_stats['avg_runs'] / (batsman_stats['run_std'] + 1e-6)


In [131]:
bowler_effect = df.groupby(['Batsman_Name_clean', 'Bowler_Bowling_Style'])['run'].mean().unstack().fillna(0)
bowler_effect.columns = [f'avg_vs_{c.replace(" ", "_").lower()}' for c in bowler_effect.columns]
batsman_stats = batsman_stats.merge(bowler_effect, left_index=True, right_index=True, how='left')


In [132]:
tod_perf = df.groupby(['Batsman_Name_clean', 'time_of_day'])['run'].mean().unstack().fillna(0)
tod_perf.columns = [f'avg_runs_{c.lower()}' for c in tod_perf.columns]
batsman_stats = batsman_stats.merge(tod_perf, left_index=True, right_index=True, how='left')


In [133]:
batsman_stats.columns

Index(['Batsman_Name_clean', 'total_runs', 'avg_runs', 'total_4s', 'total_6s',
       'total_wickets', 'total_boundaries', 'mode_pitchLine',
       'mode_pitchLength', 'balls_faced', 'strike_rate', 'boundary_rate',
       'dot_ball_pct', 'dismissal_ratio', 'avg_runs_death', 'avg_runs_middle',
       'avg_runs_powerplay', 'run_std', 'consistency_index',
       'avg_vs_left-arm_fast', 'avg_vs_left-arm_fast-medium',
       'avg_vs_left-arm_medium',
       'avg_vs_left-arm_medium,slow_left-arm_orthodox',
       'avg_vs_left-arm_medium-fast', 'avg_vs_left-arm_wrist-spin',
       'avg_vs_legbreak', 'avg_vs_legbreak_googly', 'avg_vs_right-arm_fast',
       'avg_vs_right-arm_fast-medium', 'avg_vs_right-arm_fast-medium,legbreak',
       'avg_vs_right-arm_fast-medium,legbreak_googly',
       'avg_vs_right-arm_medium', 'avg_vs_right-arm_medium,legbreak',
       'avg_vs_right-arm_medium,right-arm_offbreak',
       'avg_vs_right-arm_medium-fast',
       'avg_vs_right-arm_medium-fast,right-arm_offbr

In [134]:
batsman_stats.to_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\batsman_stats1.csv')

In [135]:
df.columns

Index(['Unnamed: 0', 'match_obj_id', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Full Name', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Full Name_bowler', 'Batting Style (s)_bowler', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role', 'Batsman_Name_clean', 'pitchLine_code',
       'pitchLength_code', 'isBoundary', 'phase'],
      dtype='object')

In [136]:
df['ball_type'] = df['pitchLength'].astype(str) + "_" + df['pitchLine'].astype(str)


In [137]:
ball_mean = df.groupby(['Batsman_Name_clean', 'ball_type'])['run'].mean().reset_index()


In [138]:
# Find the ball type with minimum mean runs (best to bowl)
best_ball = ball_mean.loc[ball_mean.groupby('Batsman_Name_clean')['run'].idxmin()]

# Rename columns for clarity
best_ball = best_ball.rename(columns={'ball_type': 'best_ball_type', 'run': 'min_mean_runs'})

# Merge with batsman stats
df_features = df_features.merge(best_ball[['Batsman_Name_clean', 'best_ball_type']], 
                                left_on='Full_Name_clean', right_on='Batsman_Name_clean', how='left')


In [139]:
df_features.head()

,Full Name,Run_Scored,Balls_Faced,Strike_Rate,Dismissals,Dismissal_Rate,Boundary_percentage,Average,Full_Name_clean,Batsman_Name_clean_x,...,total_boundaries,mode_pitchLine,mode_pitchLength,balls_faced,strike_rate,boundary_rate,dot_ball_pct,dismissal_ratio,Batsman_Name_clean_y,best_ball_type
0,AB de Villiers,117.0,61.0,191.803279,3.0,20.333333,22.950820,39.000000,ab de villiers,ab de villiers,...,14,2,2,65,180.000000,0.215385,NaN,0.046154,ab de villiers,FULL_DOWN_LEG
1,Aamir Kaleem,0.0,5.0,0.000000,2.0,2.500000,0.000000,0.000000,aamir kaleem,aamir kaleem,...,0,2,2,5,0.000000,0.000000,NaN,0.400000,aamir kaleem,FULL_OUTSIDE_OFFSTUMP
2,Aaron Finch,333.0,258.0,129.069767,11.0,23.454545,13.953488,30.272727,aaron finch,aaron finch,...,36,2,2,278,119.784173,0.129496,NaN,0.039568,aaron finch,FULL_TOSS_WIDE_OUTSIDE_OFFSTUMP
3,Aaron Johnson,101.0,73.0,138.356164,4.0,18.250000,21.917808,25.250000,aaron johnson,aaron johnson,...,16,2,2,75,134.666667,0.213333,NaN,0.053333,aaron johnson,FULL_TOSS_OUTSIDE_OFFSTUMP
4,Aaron Jones,152.0,87.0,174.712644,1.0,87.000000,21.839080,152.000000,aaron jones,aaron jones,...,19,2,2,93,163.440860,0.204301,NaN,0.010753,aaron jones,YORKER_OUTSIDE_OFFSTUMP


In [140]:
df_features.columns

Index(['Full Name', 'Run_Scored', 'Balls_Faced', 'Strike_Rate', 'Dismissals',
       'Dismissal_Rate', 'Boundary_percentage', 'Average', 'Full_Name_clean',
       'Batsman_Name_clean_x', 'total_runs', 'avg_runs', 'total_4s',
       'total_6s', 'total_wickets', 'total_boundaries', 'mode_pitchLine',
       'mode_pitchLength', 'balls_faced', 'strike_rate', 'boundary_rate',
       'dot_ball_pct', 'dismissal_ratio', 'Batsman_Name_clean_y',
       'best_ball_type'],
      dtype='object')

In [141]:
df.columns

Index(['Unnamed: 0', 'match_obj_id', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Full Name', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Full Name_bowler', 'Batting Style (s)_bowler', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role', 'Batsman_Name_clean', 'pitchLine_code',
       'pitchLength_code', 'isBoundary', 'phase', 'ball_type'],
      dtype='object')

In [142]:
batsman_stats.columns

Index(['Batsman_Name_clean', 'total_runs', 'avg_runs', 'total_4s', 'total_6s',
       'total_wickets', 'total_boundaries', 'mode_pitchLine',
       'mode_pitchLength', 'balls_faced', 'strike_rate', 'boundary_rate',
       'dot_ball_pct', 'dismissal_ratio', 'avg_runs_death', 'avg_runs_middle',
       'avg_runs_powerplay', 'run_std', 'consistency_index',
       'avg_vs_left-arm_fast', 'avg_vs_left-arm_fast-medium',
       'avg_vs_left-arm_medium',
       'avg_vs_left-arm_medium,slow_left-arm_orthodox',
       'avg_vs_left-arm_medium-fast', 'avg_vs_left-arm_wrist-spin',
       'avg_vs_legbreak', 'avg_vs_legbreak_googly', 'avg_vs_right-arm_fast',
       'avg_vs_right-arm_fast-medium', 'avg_vs_right-arm_fast-medium,legbreak',
       'avg_vs_right-arm_fast-medium,legbreak_googly',
       'avg_vs_right-arm_medium', 'avg_vs_right-arm_medium,legbreak',
       'avg_vs_right-arm_medium,right-arm_offbreak',
       'avg_vs_right-arm_medium-fast',
       'avg_vs_right-arm_medium-fast,right-arm_offbr

In [143]:
def compute_pressure(row):
    if row['oversActual'] <= 6:
        phase_mult = 1.0   
    elif row['oversActual'] <= 15:
        phase_mult = 1.2   
    else:
        phase_mult = 1.5  


    pressure = ((row['totalWickets'] + 1) / (row['oversActual'] + 1)) * phase_mult
    return pressure

df['pressure_index'] = df.apply(compute_pressure, axis=1)


In [144]:
df.columns

Index(['Unnamed: 0', 'match_obj_id', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Full Name', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Full Name_bowler', 'Batting Style (s)_bowler', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role', 'Batsman_Name_clean', 'pitchLine_code',
       'pitchLength_code', 'isBoundary', 'phase', 'ball_type',
       'pressure_index'],
      dtype='object')

In [145]:
df = df.merge(
    batsman_stats[['avg_runs']],
    left_on='Batsman_Name_clean',
    right_index=True,
    how='left'
)

df['pressure_index_batsman'] = df['pressure_index'] * (1 - (df['run'] / (df['avg_runs'] + 1e-6)))


In [146]:
df.columns

Index(['Unnamed: 0', 'match_obj_id', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Full Name', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Full Name_bowler', 'Batting Style (s)_bowler', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role', 'Batsman_Name_clean', 'pitchLine_code',
       'pitchLength_code', 'isBoundary', 'phase', 'ball_type',
       'pressure_index', 'avg_runs', 'pressure_index_batsman'],
      dtype='object')

In [147]:
df_features.columns

Index(['Full Name', 'Run_Scored', 'Balls_Faced', 'Strike_Rate', 'Dismissals',
       'Dismissal_Rate', 'Boundary_percentage', 'Average', 'Full_Name_clean',
       'Batsman_Name_clean_x', 'total_runs', 'avg_runs', 'total_4s',
       'total_6s', 'total_wickets', 'total_boundaries', 'mode_pitchLine',
       'mode_pitchLength', 'balls_faced', 'strike_rate', 'boundary_rate',
       'dot_ball_pct', 'dismissal_ratio', 'Batsman_Name_clean_y',
       'best_ball_type'],
      dtype='object')

In [148]:
batsman_stats.columns

Index(['Batsman_Name_clean', 'total_runs', 'avg_runs', 'total_4s', 'total_6s',
       'total_wickets', 'total_boundaries', 'mode_pitchLine',
       'mode_pitchLength', 'balls_faced', 'strike_rate', 'boundary_rate',
       'dot_ball_pct', 'dismissal_ratio', 'avg_runs_death', 'avg_runs_middle',
       'avg_runs_powerplay', 'run_std', 'consistency_index',
       'avg_vs_left-arm_fast', 'avg_vs_left-arm_fast-medium',
       'avg_vs_left-arm_medium',
       'avg_vs_left-arm_medium,slow_left-arm_orthodox',
       'avg_vs_left-arm_medium-fast', 'avg_vs_left-arm_wrist-spin',
       'avg_vs_legbreak', 'avg_vs_legbreak_googly', 'avg_vs_right-arm_fast',
       'avg_vs_right-arm_fast-medium', 'avg_vs_right-arm_fast-medium,legbreak',
       'avg_vs_right-arm_fast-medium,legbreak_googly',
       'avg_vs_right-arm_medium', 'avg_vs_right-arm_medium,legbreak',
       'avg_vs_right-arm_medium,right-arm_offbreak',
       'avg_vs_right-arm_medium-fast',
       'avg_vs_right-arm_medium-fast,right-arm_offbr

In [149]:
df['balls_bowled'] = (df['oversActual'].astype(int) * 6) + ((df['oversActual'] % 1) * 10)
df['remaining_balls'] = 120 - df['balls_bowled']
df['remaining_overs'] = df['remaining_balls'] / 6
df['remaining_overs'] = df['remaining_overs'].clip(lower=0)


In [150]:
batsman_context = (
    df.groupby('Batsman_Name_clean')[['oversActual', 'remaining_overs', 'pressure_index_batsman']]
    .mean()
    .reset_index()
)
batsman_stats = batsman_stats.merge(batsman_context, on='Batsman_Name_clean', how='left')
print(batsman_stats.columns.tolist())


['Batsman_Name_clean', 'total_runs', 'avg_runs', 'total_4s', 'total_6s', 'total_wickets', 'total_boundaries', 'mode_pitchLine', 'mode_pitchLength', 'balls_faced', 'strike_rate', 'boundary_rate', 'dot_ball_pct', 'dismissal_ratio', 'avg_runs_death', 'avg_runs_middle', 'avg_runs_powerplay', 'run_std', 'consistency_index', 'avg_vs_left-arm_fast', 'avg_vs_left-arm_fast-medium', 'avg_vs_left-arm_medium', 'avg_vs_left-arm_medium,slow_left-arm_orthodox', 'avg_vs_left-arm_medium-fast', 'avg_vs_left-arm_wrist-spin', 'avg_vs_legbreak', 'avg_vs_legbreak_googly', 'avg_vs_right-arm_fast', 'avg_vs_right-arm_fast-medium', 'avg_vs_right-arm_fast-medium,legbreak', 'avg_vs_right-arm_fast-medium,legbreak_googly', 'avg_vs_right-arm_medium', 'avg_vs_right-arm_medium,legbreak', 'avg_vs_right-arm_medium,right-arm_offbreak', 'avg_vs_right-arm_medium-fast', 'avg_vs_right-arm_medium-fast,right-arm_offbreak', 'avg_vs_right-arm_offbreak', 'avg_vs_right-arm_offbreak,legbreak', 'avg_vs_right-arm_offbreak,legbreak_go

In [160]:
# Calculate wicket rate per batsman and zone
wicket_rate_zone = (
    df.groupby(['Batsman_Name_clean', 'pitchLine', 'pitchLength'])['isWicket']
      .mean()
      .reset_index()
      .rename(columns={'isWicket': 'wicket_rate_zone'})
)

if 'wicket_rate_zone' in df.columns:
    df = df.drop(columns=['wicket_rate_zone'])

# Merge once only!
df = df.merge(
    wicket_rate_zone,
    on=['Batsman_Name_clean', 'pitchLine', 'pitchLength'],
    how='left'
)

# Calculate each batsman’s average wicket rate across all zones
batsman_wicket_zone_avg = (
    wicket_rate_zone.groupby('Batsman_Name_clean')['wicket_rate_zone']
      .mean()
      .reset_index()
)

# Merge with batsman_stats
batsman_stats = batsman_stats.merge(
    batsman_wicket_zone_avg, on='Batsman_Name_clean', how='left'
)

print(" wicket_rate_zone feature added successfully!")
print(df[['Batsman_Name_clean', 'pitchLine', 'pitchLength', 'wicket_rate_zone']].head())


 wicket_rate_zone feature added successfully!
  Batsman_Name_clean      pitchLine             pitchLength  wicket_rate_zone
0           tony ura  ON_THE_STUMPS             GOOD_LENGTH          0.166667
1           tony ura  ON_THE_STUMPS  SHORT_OF_A_GOOD_LENGTH          0.000000
2           tony ura  ON_THE_STUMPS             GOOD_LENGTH          0.166667
3           tony ura  ON_THE_STUMPS             GOOD_LENGTH          0.166667
4           tony ura  ON_THE_STUMPS             GOOD_LENGTH          0.166667


In [ ]:
df['phase_code'] = df['phase'].map({'Powerplay':0, 'Middle':1, 'Death':2})


In [ ]:
X_features = [

    'avg_runs', 'strike_rate', 'boundary_rate', 'dot_ball_pct',
    'dismissal_ratio', 'run_std', 'consistency_index',

    
    'avg_runs_powerplay', 'avg_runs_middle', 'avg_runs_death',

    'balls_faced', 'oversActual', 'remaining_overs', 'pressure_index_batsman',

    
    'avg_runs_day', 'avg_runs_daynight', 'avg_runs_night',


    'avg_vs_left-arm_fast', 'avg_vs_left-arm_fast-medium', 'avg_vs_left-arm_medium',
    'avg_vs_right-arm_fast', 'avg_vs_right-arm_fast-medium',
    'avg_vs_right-arm_medium', 'avg_vs_right-arm_offbreak',
    'avg_vs_slow_left-arm_orthodox',

    
    'wicket_rate_zone'
]


In [ ]:
batsman_stats.columns

Index(['Batsman_Name_clean', 'total_runs', 'avg_runs', 'total_4s', 'total_6s',
       'total_wickets', 'total_boundaries', 'mode_pitchLine',
       'mode_pitchLength', 'balls_faced', 'strike_rate', 'boundary_rate',
       'dot_ball_pct', 'dismissal_ratio', 'avg_runs_death', 'avg_runs_middle',
       'avg_runs_powerplay', 'run_std', 'consistency_index',
       'avg_vs_left-arm_fast', 'avg_vs_left-arm_fast-medium',
       'avg_vs_left-arm_medium',
       'avg_vs_left-arm_medium,slow_left-arm_orthodox',
       'avg_vs_left-arm_medium-fast', 'avg_vs_left-arm_wrist-spin',
       'avg_vs_legbreak', 'avg_vs_legbreak_googly', 'avg_vs_right-arm_fast',
       'avg_vs_right-arm_fast-medium', 'avg_vs_right-arm_fast-medium,legbreak',
       'avg_vs_right-arm_fast-medium,legbreak_googly',
       'avg_vs_right-arm_medium', 'avg_vs_right-arm_medium,legbreak',
       'avg_vs_right-arm_medium,right-arm_offbreak',
       'avg_vs_right-arm_medium-fast',
       'avg_vs_right-arm_medium-fast,right-arm_offbr

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score



In [ ]:
from sklearn.calibration import LabelEncoder


le_bowler = LabelEncoder()
df['Bowler_Bowling_Style_encoded'] = le_bowler.fit_transform(df['Bowler_Bowling_Style'].astype(str))

In [ ]:
merged = df.merge(
        batsman_stats[['Batsman_Name_clean', 'strike_rate', 'boundary_rate', 
                       'dot_ball_pct', 'dismissal_ratio', 'consistency_index', 
                       'avg_runs_powerplay', 'avg_runs_middle', 'avg_runs_death']],
        on='Batsman_Name_clean', how='left'
    )

In [ ]:
feature_cols = [
        'strike_rate', 'boundary_rate', 'dot_ball_pct', 'dismissal_ratio',
        'consistency_index', 'pressure_index_batsman',
        'avg_runs_powerplay', 'avg_runs_middle', 'avg_runs_death',
        'remaining_overs', 'inningNumber', 'oversActual', 
        'Bowler_Bowling_Style_encoded'
    ]

In [ ]:
missing = [f for f in feature_cols if f not in merged.columns]
if missing:
    raise KeyError(f"Missing expected features in merged: {missing}")

In [ ]:
bowling_types = merged['Bowler_Bowling_Style'].dropna().unique()
bowling_type_models = {}


In [155]:
for btype in bowling_types:
        subset = merged[merged['Bowler_Bowling_Style'] == btype]

        if len(subset) < 50:
            print(f"  Skipping {btype} — insufficient data ({len(subset)})")
            continue

        X = subset[feature_cols].fillna(0)
        y_line = subset['pitchLine_code']
        y_length = subset['pitchLength_code']


        # X.to_csv(r"C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\model_feature_engineered_datasets/df_model_line.csv", index=False)

        # Train line model
        X_train, X_test, y_train, y_test = train_test_split(X, y_line, test_size=0.2, random_state=42)
        model_line = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
        model_line.fit(X_train, y_train)
        acc_line = accuracy_score(y_test, model_line.predict(X_test))

        # Train length model
        X_train, X_test, y_train, y_test = train_test_split(X, y_length, test_size=0.2, random_state=42)
        model_length = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
        model_length.fit(X_train, y_train)
        acc_len = accuracy_score(y_test, model_length.predict(X_test))

        print(f" {btype} — Line: {acc_line:.3f} | Length: {acc_len:.3f}")

        bowling_type_models[btype] = {
            'line_model': model_line,
            'length_model': model_length,
            'line_acc': acc_line,
            'length_acc': acc_len
        }

 left-arm medium-fast — Line: 0.493 | Length: 0.357
 legbreak — Line: 0.518 | Length: 0.577
 right-arm offbreak,legbreak googly — Line: 0.583 | Length: 0.458
 right-arm medium-fast — Line: 0.520 | Length: 0.381
 slow left-arm orthodox — Line: 0.522 | Length: 0.616
 right-arm medium — Line: 0.564 | Length: 0.408
 left-arm medium — Line: 0.600 | Length: 0.486
  Skipping right-arm fast-medium,legbreak — insufficient data (19)
 left-arm fast — Line: 0.467 | Length: 0.287
 right-arm offbreak — Line: 0.521 | Length: 0.601
 right-arm fast — Line: 0.572 | Length: 0.342
 left-arm fast-medium — Line: 0.519 | Length: 0.391
 right-arm fast-medium — Line: 0.520 | Length: 0.327
 right-arm medium,legbreak — Line: 0.529 | Length: 0.294
 legbreak googly — Line: 0.505 | Length: 0.575
 left-arm wrist-spin — Line: 0.575 | Length: 0.475
 right-arm offbreak,legbreak — Line: 0.421 | Length: 0.447
  Skipping left-arm medium,slow left-arm orthodox — insufficient data (14)
  Skipping right-arm medium-fast,right

In [159]:
def recommend_optimal_strategy(batsman_name, phase):
    """
    Recommend the optimal bowling strategy for a given batsman and match phase.
    Uses global variables: batsman_stats, df, bowling_type_models, feature_cols,
    pitchLine_mapping, pitchLength_mapping, and le_bowler.
    """

    # Normalize inputs
    batsman_name = batsman_name.strip().lower()
    phase = phase.strip().lower()

    # Validate batsman existence
    if batsman_name not in batsman_stats['Batsman_Name_clean'].values:
        return {"Error": f"Batsman '{batsman_name}' not found in batsman_stats."}

    # Validate phase 
    phase_map = {'powerplay': 'avg_runs_powerplay', 'middle': 'avg_runs_middle', 'death': 'avg_runs_death'}
    if phase not in phase_map:
        return {"Error": f"Invalid phase '{phase}'. Choose from {list(phase_map.keys())}."}

    # Get batsman data 
    batsman_row = batsman_stats[batsman_stats['Batsman_Name_clean'] == batsman_name].iloc[0]

    # Build contextual sample
    # These values are derived dynamically from df for realism
    df_bat = df[df['Batsman_Name_clean'] == batsman_name]
    sample_data = {
        'strike_rate': batsman_row['strike_rate'],
        'boundary_rate': batsman_row['boundary_rate'],
        'dot_ball_pct': batsman_row['dot_ball_pct'],
        'dismissal_ratio': batsman_row['dismissal_ratio'],
        'consistency_index': batsman_row['consistency_index'],
        'pressure_index_batsman': df_bat['pressure_index_batsman'].mean() if 'pressure_index_batsman' in df_bat else 0,
        'avg_runs_powerplay': batsman_row['avg_runs_powerplay'],
        'avg_runs_middle': batsman_row['avg_runs_middle'],
        'avg_runs_death': batsman_row['avg_runs_death'],
        'remaining_overs': df_bat['remaining_overs'].mean() if 'remaining_overs' in df_bat else 10,
        'inningNumber': df_bat['inningNumber'].mode()[0] if not df_bat['inningNumber'].mode().empty else 1,
        'oversActual': df_bat['oversActual'].mean() if 'oversActual' in df_bat else 10
    }

    results = []

    #  Iterate through trained bowling types 
    for btype, models in bowling_type_models.items():
        try:
            # Encode bowling style for model input
            encoded_style = le_bowler.transform([btype])[0]
            sample_data['Bowler_Bowling_Style_encoded'] = encoded_style

            # Build dataframe for prediction
            sample_df = pd.DataFrame([sample_data])[feature_cols].fillna(0)

            # Predict pitch line and length
            line_pred = models['line_model'].predict(sample_df)[0]
            length_pred = models['length_model'].predict(sample_df)[0]

            # Decode predictions
            decoded_line = pitchLine_mapping.get(int(line_pred), f"Unknown({line_pred})")
            decoded_length = pitchLength_mapping.get(int(length_pred), f"Unknown({length_pred})")

            results.append({
                'Bowling Type': btype,
                'Predicted Line': decoded_line,
                'Predicted Length': decoded_length,
                'Line Model Accuracy': round(models['line_acc'], 3),
                'Length Model Accuracy': round(models['length_acc'], 3)
            })

        except Exception as e:
            print(f" Skipping {btype}: {e}")
            continue

    if not results:
        return {"Error": f"No valid predictions available for {batsman_name}."}

    return {
        'Batsman': batsman_name.title(),
        'Phase': phase.title(),
        'Recommendations': results
    }


In [158]:
result = recommend_optimal_strategy('rohit sharma', 'middle')
result


{'Batsman': 'Rohit Sharma',
 'Phase': 'Middle',
 'Recommendations': [{'Bowling Type': 'left-arm medium-fast',
   'Predicted Line': 'ON_THE_STUMPS',
   'Predicted Length': 'GOOD_LENGTH',
   'Line Model Accuracy': 0.493,
   'Length Model Accuracy': 0.357},
  {'Bowling Type': 'legbreak',
   'Predicted Line': 'OUTSIDE_OFFSTUMP',
   'Predicted Length': 'GOOD_LENGTH',
   'Line Model Accuracy': 0.518,
   'Length Model Accuracy': 0.577},
  {'Bowling Type': 'right-arm offbreak,legbreak googly',
   'Predicted Line': 'ON_THE_STUMPS',
   'Predicted Length': 'GOOD_LENGTH',
   'Line Model Accuracy': 0.583,
   'Length Model Accuracy': 0.458},
  {'Bowling Type': 'right-arm medium-fast',
   'Predicted Line': 'OUTSIDE_OFFSTUMP',
   'Predicted Length': 'GOOD_LENGTH',
   'Line Model Accuracy': 0.52,
   'Length Model Accuracy': 0.381},
  {'Bowling Type': 'slow left-arm orthodox',
   'Predicted Line': 'ON_THE_STUMPS',
   'Predicted Length': 'GOOD_LENGTH',
   'Line Model Accuracy': 0.522,
   'Length Model Ac

In [ ]:
df['pitchLength'].unique()

array(['GOOD_LENGTH', 'SHORT_OF_A_GOOD_LENGTH', 'SHORT', 'FULL', 'YORKER',
       'FULL_TOSS'], dtype=object)

In [ ]:
length_to_ball_type = {
    'YORKER': 'yorker',
    'FULL': 'full',
    'FULL_TOSS': 'full',
    'GOOD_LENGTH': 'good',
    'SHORT_OF_A_GOOD_LENGTH': 'short',
    'SHORT': 'short'
}

df['ball_type'] = df['pitchLength'].map(length_to_ball_type)

pitchLength	count
GOOD_LENGTH	1800
SHORT	1400
FULL	800
YORKER	400
FULL_TOSS	60

then FULL_TOSS has too few samples (only 60) to train a stable class.

✅ So we group it with “FULL” because:

In [ ]:
df['ball_type'].value_counts()


ball_type
good      14524
short      9759
full       7890
yorker      856
Name: count, dtype: int64

In [ ]:
df_merged = df.merge(
    batsman_stats[[
        'Batsman_Name_clean', 'strike_rate', 'boundary_rate', 'dot_ball_pct',
        'dismissal_ratio', 'consistency_index', 'pressure_index_batsman',
        'avg_runs_powerplay', 'avg_runs_middle', 'avg_runs_death'
    ]],
    on='Batsman_Name_clean', how='left'
)

In [ ]:
df_merged = df_merged.merge(
        df[['Batsman_Name_clean', 'pressure_index_batsman']].drop_duplicates(),
        on='Batsman_Name_clean', how='left'
    )

In [ ]:
le_balltype = LabelEncoder()
df_merged['ball_type_encoded'] = le_balltype.fit_transform(df_merged['ball_type'].astype(str))


In [ ]:
feature_cols_balltype = [
    'strike_rate', 'boundary_rate', 'dot_ball_pct', 'dismissal_ratio',
    'consistency_index', 'pressure_index_batsman',
    'avg_runs_powerplay', 'avg_runs_middle', 'avg_runs_death',
    'remaining_overs', 'oversActual', 'inningNumber'
]

In [ ]:
df_model = df_merged.dropna(subset=['ball_type_encoded'])

In [ ]:
X = df_model[feature_cols_balltype].fillna(0)
y = df_model['ball_type_encoded']
X.to_csv(r"C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\model_feature_engineered_datasets/df_model_ball_type.csv", index=False)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model_balltype = RandomForestClassifier(n_estimators=150, random_state=42, max_depth=12)
model_balltype.fit(X_train, y_train)

RandomForestClassifier(max_depth=12, n_estimators=150, random_state=42)

In [ ]:
y_pred = model_balltype.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f" Ball Type Model Accuracy: {acc:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le_balltype.classes_))

✅ Ball Type Model Accuracy: 0.450

Classification Report:
              precision    recall  f1-score   support

        full       0.35      0.13      0.19      1566
        good       0.47      0.84      0.60      2897
       short       0.40      0.17      0.24      1961
      yorker       0.09      0.01      0.01       182

    accuracy                           0.45      6606
   macro avg       0.33      0.29      0.26      6606
weighted avg       0.41      0.45      0.38      6606



In [ ]:
def recommend_best_ball_type(batsman_name, phase):
    batsman_name = batsman_name.lower().strip()
    sample = df_merged[
        (df_merged['Batsman_Name_clean'] == batsman_name) &
        (df_merged['phase'].str.lower() == phase.lower())
    ]
    
    if sample.empty:
        sample = df_merged[df_merged['Batsman_Name_clean'] == batsman_name].sample(1)
    
    X_sample = sample[feature_cols_balltype].fillna(0)
    pred_class = model_balltype.predict(X_sample)
    return le_balltype.inverse_transform(pred_class)[0]

# Example:
print("🎯 Recommended Ball Type:", recommend_best_ball_type("MS Dhoni", "Middle"))


🎯 Recommended Ball Type: good


In [ ]:
df['phase'].unique()

array(['Powerplay', 'Middle', 'Death'], dtype=object)

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(model_line, "models/pitch_line_model.joblib")
print("✅ Pitch line model saved.")

✅ Pitch line model saved.


In [ ]:
joblib.dump(model_length,r"C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\models\pitch_length_model.joblib")


['C:\\Users\\yaswa\\OneDrive\\Desktop\\artificial intelligence project\\models\\pitch_length_model.joblib']

In [ ]:
joblib.dump(model_balltype,r"C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\models\model_balltype.joblib")

['C:\\Users\\yaswa\\OneDrive\\Desktop\\artificial intelligence project\\models\\model_balltype.joblib']